In [1]:
!pip install -q transformers trl peft accelerate bitsandbytes flash-attn --no-build-isolation

In [2]:
import torch
from transformers import AutoModelForCausalLM, AutoTokenizer
from trl import DPOConfig, DPOTrainer
from peft import LoraConfig, get_peft_model, prepare_model_for_kbit_training
from datasets import Dataset

print(f"PyTorch version: {torch.__version__}")
print(f"CUDA available: {torch.cuda.is_available()}")
print(f"GPU: {torch.cuda.get_device_name(0) if torch.cuda.is_available() else 'None'}")

PyTorch version: 2.8.0+cu126
CUDA available: True
GPU: Tesla T4


Here's how to add your `HF_TOKEN` secret in Colab:

1.  On the left sidebar of your Colab notebook, click on the **key icon** (Secrets).
2.  Click on **+ New secret**.
3.  In the **Name** field, type `HF_TOKEN`.
4.  In the **Value** field, paste your Hugging Face User Access Token.
5.  Make sure the **Notebook access** toggle is turned **ON** for this notebook.
6.  Click **Done**.

After adding the secret, you might need to restart your Colab runtime for the changes to take effect.

In [3]:
# ===== LOAD MODEL (SMALLER FOR COLAB) =====
# Use Llama-3.2-1B instead of 8B (fits in T4 16GB)
model_id = "meta-llama/Llama-3.2-1B"

print("\n📦 Loading model with Flash Attention 2...")

# Retrieve HF token from Colab secrets
from google.colab import userdata
hf_token = userdata.get('HF_TOKEN')

try:
    model = AutoModelForCausalLM.from_pretrained(
        model_id,
        torch_dtype=torch.bfloat16,
        device_map="auto",
        attn_implementation="flash_attention_2",  # ✅ Testing this
        token=hf_token  # Use the token from secrets
    )
    print("✅ Model loaded successfully with Flash Attention 2")
except Exception as e:
    print(f"❌ Model loading failed: {e}")
    exit(1)

tokenizer = AutoTokenizer.from_pretrained(model_id, token=hf_token) # Use the token for tokenizer as well
tokenizer.pad_token = tokenizer.eos_token
tokenizer.model_max_length = 512  # Short for testing


📦 Loading model with Flash Attention 2...


`torch_dtype` is deprecated! Use `dtype` instead!


✅ Model loaded successfully with Flash Attention 2


tokenizer_config.json:   0%|          | 0.00/50.5k [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/9.09M [00:00<?, ?B/s]

special_tokens_map.json:   0%|          | 0.00/301 [00:00<?, ?B/s]

In [4]:
# ===== APPLY LORA =====
print("\n🔧 Applying LoRA...")
model = prepare_model_for_kbit_training(model, use_gradient_checkpointing=True)

lora_config = LoraConfig(
    r=8,  # Smaller for testing
    lora_alpha=8,
    target_modules=["q_proj", "k_proj", "v_proj", "o_proj"],
    lora_dropout=0.0,
    bias="none",
    task_type="CAUSAL_LM",
)
model = get_peft_model(model, lora_config)
model.print_trainable_parameters()


🔧 Applying LoRA...
trainable params: 1,703,936 || all params: 1,237,518,336 || trainable%: 0.1377


In [5]:
# ===== CREATE DUMMY DATASET =====
print("\n📊 Creating dummy dataset...")
dummy_data = {
    "prompt": ["Hello, how are you?"] * 8,
    "chosen": ["I'm doing great, thank you!"] * 8,
    "rejected": ["I don't know."] * 8,
}
dataset = Dataset.from_dict(dummy_data)


📊 Creating dummy dataset...


In [9]:
# ===== TEST 1: WITHOUT ref_model_dtype (expected to fail?) =====
print("\n" + "="*80)
print("TEST 1: DPOConfig WITHOUT ref_model_dtype (baseline)")
print("="*80)

training_args_v1 = DPOConfig(
    output_dir="./test_output_v1",
    beta=0.1,
    per_device_train_batch_size=1,
    max_steps=5,  # Just 5 steps for testing
    learning_rate=2e-5,
    logging_steps=1,
    bf16=True,
    # NO ref_model_dtype specified
    report_to="none", # Skip wandb
)

try:
    trainer_v1 = DPOTrainer(
        model=model,
        ref_model=None,  # DPOTrainer creates reference model internally
        args=training_args_v1,
        train_dataset=dataset,
        processing_class=tokenizer,
    )
    print("✅ DPOTrainer initialized successfully")

    print("\n🏃 Running 5 training steps...")
    trainer_v1.train()
    print("✅ TEST 1 PASSED: Training completed without dtype errors")

except RuntimeError as e:
    if "dtype" in str(e).lower() or "bfloat16" in str(e).lower():
        print(f"❌ TEST 1 FAILED: Dtype error (expected)")
        print(f"   Error: {e}")
    else:
        print(f"❌ TEST 1 FAILED: Unexpected error")
        print(f"   Error: {e}")

except Exception as e:
    print(f"❌ TEST 1 FAILED: {e}")


TEST 1: DPOConfig WITHOUT ref_model_dtype (baseline)


Extracting prompt in train dataset:   0%|          | 0/8 [00:00<?, ? examples/s]

Applying chat template to train dataset:   0%|          | 0/8 [00:00<?, ? examples/s]

Tokenizing train dataset:   0%|          | 0/8 [00:00<?, ? examples/s]

The model is already on multiple devices. Skipping the move to device specified in `args`.


✅ DPOTrainer initialized successfully

🏃 Running 5 training steps...


Casting fp32 inputs back to torch.bfloat16 for flash-attn compatibility.


❌ TEST 1 FAILED: Unexpected error
   Error: FlashAttention only supports Ampere GPUs or newer.


In [11]:
# ===== TEST 2: WITH ref_model_dtype (testing fix) =====
print("\n" + "="*80)
print("TEST 2: DPOConfig WITH ref_model_dtype=torch.bfloat16 (proposed fix)")
print("="*80)

training_args_v2 = DPOConfig(
    output_dir="./test_output_v2",
    beta=0.1,
    per_device_train_batch_size=1,
    max_steps=5,
    learning_rate=2e-5,
    logging_steps=1,
    bf16=True,
    # ✅ TESTING THIS FIX - Removed ref_model_dtype as it's not supported
    # ref_model_dtype=torch.bfloat16,  # Force reference model to use BF16
    # TO AVOID: TypeError: DPOConfig.__init__() got an unexpected keyword argument 'ref_model_dtype'
    report_to="none", # Skip wandb
)

try:
    # Reload model to avoid state from test 1
    # Retrieve HF token from Colab secrets
    from google.colab import userdata
    hf_token = userdata.get('HF_TOKEN')

    model2 = AutoModelForCausalLM.from_pretrained(
        model_id,
        torch_dtype=torch.bfloat16,
        device_map="auto",
        attn_implementation="flash_attention_2",
        token=hf_token # Use HF token from secrets
    )
    model2 = prepare_model_for_kbit_training(model2, use_gradient_checkpointing=True)
    model2 = get_peft_model(model2, lora_config)

    trainer_v2 = DPOTrainer(
        model=model2,
        ref_model=None,
        args=training_args_v2,
        train_dataset=dataset,
        processing_class=tokenizer,
    )
    print("✅ DPOTrainer initialized successfully")

    print("\n🏃 Running 5 training steps...")
    trainer_v2.train()
    print("✅ TEST 2 PASSED: Training completed with ref_model_dtype fix")

except RuntimeError as e:
    if "dtype" in str(e).lower():
        print(f"❌ TEST 2 FAILED: Dtype error persists (fix didn't work)")
        print(f"   Error: {e}")
    else:
        print(f"❌ TEST 2 FAILED: Unexpected error")
        print(f"   Error: {e}")

except Exception as e:
    print(f"❌ TEST 2 FAILED: {e}")


TEST 2: DPOConfig WITH ref_model_dtype=torch.bfloat16 (proposed fix)


Extracting prompt in train dataset:   0%|          | 0/8 [00:00<?, ? examples/s]

Applying chat template to train dataset:   0%|          | 0/8 [00:00<?, ? examples/s]

Tokenizing train dataset:   0%|          | 0/8 [00:00<?, ? examples/s]

The model is already on multiple devices. Skipping the move to device specified in `args`.
The tokenizer has new PAD/BOS/EOS tokens that differ from the model config and generation config. The model config and generation config were aligned accordingly, being updated with the tokenizer's values. Updated tokens: {'pad_token_id': 128001}.


✅ DPOTrainer initialized successfully

🏃 Running 5 training steps...
❌ TEST 2 FAILED: Unexpected error
   Error: FlashAttention only supports Ampere GPUs or newer.


In [12]:
# ===== SUMMARY =====
print("\n" + "="*80)
print("📊 TEST SUMMARY")
print("="*80)
print("Test 1 (no ref_model_dtype): Check output above")
print("Test 2 (with ref_model_dtype): Check output above")
print("\nIf both tests pass, Flash Attention 2 works without dtype fix.")
print("If Test 1 fails but Test 2 passes, the fix works!")
print("If both fail, Flash Attention 2 is incompatible with CITA/DPO.")
print("="*80)

# ===== MEMORY REPORT =====
if torch.cuda.is_available():
    print(f"\n📊 GPU Memory Usage:")
    print(f"  Allocated: {torch.cuda.memory_allocated() / 1024**3:.2f} GB")
    print(f"  Reserved:  {torch.cuda.memory_reserved() / 1024**3:.2f} GB")


📊 TEST SUMMARY
Test 1 (no ref_model_dtype): Check output above
Test 2 (with ref_model_dtype): Check output above

If both tests pass, Flash Attention 2 works without dtype fix.
If Test 1 fails but Test 2 passes, the fix works!
If both fail, Flash Attention 2 is incompatible with CITA/DPO.

📊 GPU Memory Usage:
  Allocated: 9.23 GB
  Reserved:  9.49 GB
